# Entrega 3 · Ejecución completa en Google Colab

**Proyecto:** Predicción e interpretación de imágenes de ECG con modelos de visión-lenguaje · Alfonso Gysling · CINF200

Antes de empezar:
1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar.**
2. Ten el dataset de Mendeley Data (v2) en tu Google Drive, como ZIP o como carpeta descomprimida.
3. Ejecuta las celdas **en orden**. Los resultados y modelos se guardan en tu Drive: si Colab se desconecta, vuelve a ejecutar desde la celda 1 y las corridas ya terminadas se omiten solas.

Tiempo aproximado en una T4: 2 a 3 horas en total (BLIP es lo más lento).

## 1 · Configuración (edita solo esta celda)

In [ ]:
REPO_URL  = "https://github.com/gyslinga-debug/INTEGRADOR-II-SEM-LIC-EN-CS-Alfonso-Gysling.git"
DATA_DIR  = "/content/drive/MyDrive/ECG_dataset"        # ZIP de Mendeley o carpeta con las 4 carpetas de imágenes
DRIVE_OUT = "/content/drive/MyDrive/ecg-vlm-salidas"    # aquí quedan results/ y models/ (sobreviven a desconexiones)
SEEDS     = [42, 43, 44]

## 2 · GPU, Drive y repositorio

In [ ]:
!nvidia-smi -L || echo "⚠️ No hay GPU: activa T4 en Entorno de ejecución → Cambiar tipo de entorno de ejecución"
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, getpass
REPO_DIR = "/content/ecg-vlm"
if not os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print("El repositorio es privado. Pega un token de GitHub con permiso 'repo'")
        print("(GitHub → Settings → Developer settings → Personal access tokens).")
        token = getpass.getpass("Token de GitHub: ")
        subprocess.run(["git", "clone", REPO_URL.replace("https://", f"https://{token}@"), REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"])
%cd {REPO_DIR}
!ls

## 3 · Instalación y pruebas

In [ ]:
!pip install -q -r requirements.txt && pip install -q -e .
import nltk
nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
os.makedirs(f"{DRIVE_OUT}/results", exist_ok=True)
!pytest -q 2>&1 | tee "{DRIVE_OUT}/results/pytest_salida.txt"

## 4 · Datos: copiar a disco local y crear la configuración

In [ ]:
import shutil, glob, yaml
RAW_DIR = "/content/data_raw"
if not os.path.exists(RAW_DIR):
    if DATA_DIR.lower().endswith(".zip"):
        !unzip -q -o "{DATA_DIR}" -d "{RAW_DIR}"
    else:
        shutil.copytree(DATA_DIR, RAW_DIR)
n_imgs = len([p for p in glob.glob(f"{RAW_DIR}/**/*", recursive=True) if p.lower().endswith((".jpg", ".jpeg", ".png"))])
print("Imágenes encontradas:", n_imgs)

cfg = yaml.safe_load(open("configs/default.yaml"))
cfg["paths"] = {"raw_dir": RAW_DIR, "work_dir": "/content/work",
                "results_dir": f"{DRIVE_OUT}/results", "models_dir": f"{DRIVE_OUT}/models"}
cfg["seeds"] = SEEDS
yaml.safe_dump(cfg, open("configs/colab.yaml", "w"), allow_unicode=True, sort_keys=False)
os.environ["ECGVLM_CONFIG"] = os.path.abspath("configs/colab.yaml")
R, M = cfg["paths"]["results_dir"], cfg["paths"]["models_dir"]
print("Resultados en:", R, "| Modelos en:", M)

## 5 · Auditoría, preprocesamiento y partición

In [ ]:
!python -m ecgvlm.preprocess
import json
rep = json.load(open(f"{R}/audit_report.json"))
print(json.dumps({k: rep[k] for k in ["total_files", "counts_by_class_files", "exact_duplicates_removed",
                                      "kept_images", "counts_by_class_kept"]}, indent=1, ensure_ascii=False))
print("Casi-duplicados:", json.dumps(rep["near_duplicates"], ensure_ascii=False)[:600])
print("Conflictos de etiqueta (casi idénticas con clase distinta):", len(rep["near_duplicate_label_conflicts"]))
from IPython.display import Image as IPImage, display
display(IPImage(f"{R}/audit_hamming_hist.png"))

**Revisa el recorte:** abajo se muestra una imagen original y su versión procesada por clase. La cabecera de texto impreso
debe quedar fuera. Si no, ajusta `crop_top` en `configs/colab.yaml` y repite la celda 5 con `--force`.

In [ ]:
import pandas as pd
from PIL import Image
man = pd.read_csv("/content/work/manifest.csv")
for label, g in man.groupby("label"):
    r = g.iloc[0]
    orig = Image.open(os.path.join(RAW_DIR, r["rel_path"])).convert("RGB")
    orig.thumbnail((640, 640)); proc = Image.open(r["proc_path"]).resize((320, 320))
    canvas = Image.new("RGB", (orig.width + 330, max(orig.height, 320)), "white")
    canvas.paste(orig, (0, 0)); canvas.paste(proc, (orig.width + 10, 0))
    print(label); display(canvas)

## 6 · Líneas base: B2 ViT-B/16 y B3 ViT + plantilla (3 semillas)

In [ ]:
for s in SEEDS:
    !python -m ecgvlm.train_classifier --model vit_b16 --seed {s} --skip_existing

## 7 · Línea base B1 ResNet50 (3 semillas)

In [ ]:
for s in SEEDS:
    !python -m ecgvlm.train_classifier --model resnet50 --seed {s} --skip_existing

## 8 · Validación cruzada de 5 particiones (ViT)

In [ ]:
if not os.path.exists(f"{R}/vit_b16_cv5/cv_summary.json"):
    !python -m ecgvlm.train_classifier --model vit_b16 --cv 5 --seed 42
else:
    print("Validación cruzada ya realizada")

## 9 · B4: BLIP sin ajustar

In [ ]:
!python -m ecgvlm.train_blip --zero_shot --skip_existing

## 10 · P: BLIP ajustado (3 semillas)
Si aparece `CUDA out of memory`, agrega `--freeze_vision` al comando (y decláralo en el informe).

In [ ]:
for s in SEEDS:
    !python -m ecgvlm.train_blip --seed {s} --skip_existing

## 11 · Evaluación de texto, McNemar y planilla de revisión cualitativa

In [ ]:
for s in SEEDS:
    !python -m ecgvlm.evaluate --pred "{R}/blip_ft_s{s}/pred_test.csv" --text
    !python -m ecgvlm.evaluate --compare "{R}/vit_b16_s{s}/pred_test.csv" "{R}/blip_ft_s{s}/pred_test.csv"
!python -m ecgvlm.evaluate --pred "{R}/vit_b16_template_s42/pred_test.csv" "{R}/blip_zs/pred_test.csv"
!python -m ecgvlm.evaluate --review "{R}/blip_ft_s42/pred_test.csv"

## 12 · Robustez (8 perturbaciones) y Grad-CAM

In [ ]:
!python -m ecgvlm.robustness --classifier "{M}/vit_b16_s42.pt" "{M}/resnet50_s42.pt" --blip "{M}/blip_ft_s42"
!python -m ecgvlm.gradcam --classifier "{M}/vit_b16_s42.pt" --per_class 5
!python -m ecgvlm.gradcam --classifier "{M}/resnet50_s42.pt" --per_class 5

## 13 · Tabla resumen y criterios

In [ ]:
!python -m ecgvlm.summarize > /dev/null
from IPython.display import Markdown
display(Markdown(open(f"{R}/summary.md", encoding="utf-8").read()))
display(pd.read_csv(f"{R}/robustness.csv").pivot_table(index="perturbation", columns="model", values="delta_pp").round(1))

## 14 · Revisión manual (fuera de Colab) y cierre
1. Descarga y completa `results/blip_ft_s42/review_test.csv` (puntajes 0 a 2 en las 40 salidas).
2. Completa la columna `en_trazado_manual` (si/no) de `results/gradcam_vit_b16_s42/gradcam_review.csv` mirando los mapas.
3. Súbelos de vuelta a la misma carpeta de Drive y ejecuta la celda siguiente para recalcular los criterios.

In [ ]:
if os.path.exists(f"{R}/blip_ft_s42/review_test.csv"):
    !python -m ecgvlm.evaluate --score_review "{R}/blip_ft_s42/review_test.csv"
!python -m ecgvlm.summarize > /dev/null
display(Markdown(open(f"{R}/summary.md", encoding="utf-8").read()))

## 15 · Descargar los resultados (ZIP, sin los modelos)

In [ ]:
from google.colab import files
zip_base = "/content/resultados_e3"
shutil.make_archive(zip_base, "zip", root_dir=os.path.dirname(R), base_dir=os.path.basename(R))
print(f"{os.path.getsize(zip_base + '.zip') / 1e6:.1f} MB")
files.download(zip_base + ".zip")

## 16 · Subir los resultados a GitHub
Copia `results/` (CSV, JSON y PNG; nunca los modelos) al repositorio y hace commit y push.
Necesitas un token de GitHub con permiso `repo`.

In [ ]:
GIT_NAME  = "Alfonso Gysling"
GIT_EMAIL = "tu_correo@ejemplo.com"     # el correo de tu cuenta de GitHub
token = getpass.getpass("Token de GitHub: ")
dst = os.path.join(REPO_DIR, "results")
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(R, dst, ignore=shutil.ignore_patterns("*.pt", "*.bin", "*.safetensors"))
!git -C "{REPO_DIR}" config user.name "{GIT_NAME}"
!git -C "{REPO_DIR}" config user.email "{GIT_EMAIL}"
!git -C "{REPO_DIR}" add results
!git -C "{REPO_DIR}" commit -m "Resultados de la Entrega 3 (Colab, GPU T4)"
push_url = REPO_URL.replace("https://", f"https://{token}@")
!git -C "{REPO_DIR}" push "{push_url}" HEAD:main
del token

## 17 · Demo (al final)
Abre el enlace público `gradio.live`, toma la captura para el informe y detén la celda con ■ al terminar.
El enlace es temporal: úsalo solo para la presentación.

In [ ]:
!python -m ecgvlm.app --classifier "{M}/vit_b16_s42.pt" --blip "{M}/blip_ft_s42" --share